In [1]:
from pathlib import Path

import polars as pl

CSV_PATH = Path("data/researchers.csv")


def load_researcher_ids(csv_path: Path = CSV_PATH) -> list[int]:
    """Load researcher IDs from a CSV after validating format and contents.

    :param csv_path: Path to the CSV file containing researcher data.
    :return: List of researcher IDs.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")

    df = pl.read_csv(csv_path)

    if "idPersonal" not in df.columns:
        raise ValueError(
            f"The CSV does not contain the 'idPersonal' column. Columns found: {df.columns}"
        )

    df = (
        df.select(["idPersonal", "rfSignature"])
        .drop_nulls()
        .filter(pl.col("idPersonal").cast(pl.Int64) > 0)
        .unique()
        .sort("idPersonal")
    )

    ids = df["idPersonal"].to_list()
    names = df["rfSignature"].to_list()
    return ids, names


ids, names = load_researcher_ids()
researchers = dict(zip(ids, names))
researchers

{258423: 'HABER GUERRA, RODOLFO ELIAS',
 258729: 'ROSADO MARIA, MARIA EUGENIA',
 258829: 'VAZQUEZ GALLO, MARIA JESUS',
 260726: 'RICO ALMODOVAR, MARIANO',
 260781: 'LOSADA GONZALEZ, JUAN CARLOS',
 260874: 'PEREZ TEJERO, JAVIER',
 261274: 'CAMACHO FERNANDEZ, DAVID',
 261292: 'MENDO TOMAS, LUIS',
 261323: 'DUQUE CAMPAYO, DANIEL',
 261708: 'RUIZ CRUZ, JORGE ALFONSO',
 262424: 'LOPEZ HERNANDEZ, ISABEL',
 262511: 'SALGADO ALVAREZ DE SOTOMAYOR, LUIS',
 262522: 'FERNANDEZ GONZALEZ, JOSE MANUEL',
 262523: 'BELLO ORGAZ, GEMA',
 262605: 'MARTINEZ SAEZ, ANTONIO',
 263256: 'RAMOS GOMEZ, MARIA MILAGROS',
 264089: 'TORIBIO LOPEZ, FRANCISCO RENE',
 264099: 'ANGULO ARDOY, PABLO',
 264517: 'AGUIRRE DE CARCER GARCIA, IŃIGO',
 264698: 'MERCHAN RUBIRA, SUSANA',
 264784: 'SANCHEZ OLIVARES, PABLO',
 264906: 'GARCIA MUŃOZ, PATRICIA',
 265232: 'GONZALEZ ALVARO, DAVID',
 271169: 'BERROCAL MENARGUEZ, ANA BELEN',
 271668: 'SALAS NATERA, MIGUEL ALEJANDRO',
 271791: 'MEROŃO MORENO, CRISTOBAL',
 272078: 'GALVEZ PER

In [2]:
import json
import re
import tqdm

BASE_PATH = Path("./data/raw")


def load_ajax_dataSet(file):
    with open(file, encoding="utf-8") as f:
        ajax_html = f.read()

    match = re.search(r"var dataSet\s*=\s*(\[.*\]);", ajax_html, re.DOTALL)

    if not match:
        return pl.DataFrame()

    try:
        json_str = match.group(1)
        activity_data = json.loads(json_str)
    except json.JSONDecodeError:
        try:
            json_str = json_str.rsplit("]", 1)[0] + "]"
            activity_data = json.loads(json_str)
        except:
            return pl.DataFrame()

    if not activity_data:
        return pl.DataFrame()

    all_fields = set().union(*(record.keys() for record in activity_data))
    normalized_records = [
        {f: record.get(f) for f in all_fields} for record in activity_data
    ]

    df = pl.DataFrame(normalized_records, infer_schema_length=len(normalized_records))

    mapeo_columnas = {
        "rfTitle": "TÍTULO",
        "rfAuthor": "AUTORES",
        "rfStartDate": "FECHA INICIO",
        "rfEndDate": "FECHA FIN",
        "rfHash": "IDENTIFICADOR",
        "stOpenAccess": "OPENACCESS",
        "dtWorkform": "TIPO",
        "dtType": "TIPO DE PRODUCCIÓN",
        "rfSubject": "CATEGORÍAS",
        "rfLinkName": "FUENTE",
        "rfQuality11": "IF SJR",
        "rfQuality17": "CITAS EUROPEPMC",
        "rfQuality18": "CITAS INSPIRE",
        "rfQuality19": "CITAS SCHOLAR",
        "rfQuality22": "Q SJR",
        "rfRef": "CITACIÓN",
        "idQuality63": "PIDs",
        "rfRefUrl": "URL",
    }

    columnas_presentes = [c for c in mapeo_columnas.keys() if c in df.columns]
    if not columnas_presentes or "dtWorkform" not in columnas_presentes:
        return pl.DataFrame()

    mapeo_final = {k: mapeo_columnas[k] for k in columnas_presentes}

    df_final = df.select(columnas_presentes).rename(mapeo_final)
    sort_cols = [c for c in ["TIPO", "FECHA INICIO"] if c in df_final.columns]
    if sort_cols:
        df_final = df_final.sort(sort_cols, descending=True)
    df_final = df_final.with_columns(
        pl.col(pl.String)
        .str.replace_all(r"\r?\n", "; ")
        .str.replace_all(r";", ",")
        .str.replace_all(r"\s+", " ")
    )

    if "URL" in df_final.columns:
        df_final = df_final.with_columns(
            pl.col("URL").map_elements(
                lambda s: ", ".join(re.findall(r"https?://\S+", s)) if s else s,
                return_dtype=pl.String,
            )
        )

    return df_final


try:
    todos_los_tipos = set()
    for id in tqdm.tqdm(
        ids,
        desc="Procesando investigadores",
        unit="investigador",
    ):
        ajax_path = BASE_PATH / f"{id}_ajax.html"
        df_ajax = load_ajax_dataSet(ajax_path)
        if df_ajax.is_empty() or "TIPO" not in df_ajax.columns:
            continue
        tipos = df_ajax["TIPO"].unique().to_list()
        todos_los_tipos.update(tipos)
finally:
    print("Tipos de actividad encontrados:")
    for tipo in sorted(todos_los_tipos):
        print(f"- {tipo}")

Procesando investigadores:   0%|          | 0/5722 [00:00<?, ?investigador/s]

Procesando investigadores:   0%|          | 6/5722 [00:08<2:17:51,  1.45s/investigador]


Tipos de actividad encontrados:
- Breve Presentación
- Dirección de Tesis, TFG, TFM, etc.
- Docencia Impartida
- Estancias
- Eventos para la Divulgación
- Experiencia en Evaluación
- Experiencia profesional
- Formación
- Gestión
- Grupos, Redes, Sociedades, Foros y Comités
- Idiomas
- Materiales de lectura
- Otras Actividades Docentes
- Perfil Temático
- Propiedad Industrial e Intelectual
- Proyectos I+D+I y Ayudas
- Proyectos de Innovacion Docente
- Publicaciones
- Publicaciones Docentes
- Publicaciones de Divulgación
- Reconocimientos
- Trabajos presentados en Congresos u Otros Eventos


KeyboardInterrupt: 

In [ ]:
data_researcher = pl.read_json("xx_full_activity_carlos.json", infer_schema_length=1000)
publis = (
    data_researcher.filter(pl.col("dtWorkform") == "Publicaciones")
    .filter(pl.col("dsDescription").is_not_null())
    .select(
        pl.col("idItem"),
        pl.col("dtSubtype"),
        pl.col("rfValidationAgency"),
        pl.col("rfTitle"),
        pl.col("dsDescription"),
        # pl.col("dsDescription2"),
        # pl.col("dsDescription3"),
        # pl.col("dsDescription4"),
        pl.col("dsDescription5"),
        pl.col("rfKeyword"),
    )
)
publis.to_dicts()

[{'idItem': '10429991',
  'dtSubtype': 'Conferencia Publicada',
  'rfTitle': 'Enhancing Public Procurement in the European Union Through Constructing and Exploiting an Integrated Knowledge Graph',
  'dsDescription': "Soylu A., SINTEF Foundation for Scientific and Industrial Research; Corcho O., Universidad Politécnica de Madrid; Elvesæter B., SINTEF Foundation for Scientific and Industrial Research; Badenes-Olmedo C., Universidad Politécnica de Madrid; Martínez F.Y., Universidad Politécnica de Madrid; Kovacic M., Jozef Stefan Institute; Posinkovic M., Jozef Stefan Institute; Makgill I., OpenOpps Ltd; Taggart C., OpenCorporates Ltd; Simperl E., King's College London; Lech T.C., SINTEF Foundation for Scientific and Industrial Research; Roman D., SINTEF Foundation for Scientific and Industrial Research (Authors with Filk); Public procurement is a large market affecting almost every organisation and individual. Governments need to ensure efficiency, transparency, and accountability, while 

In [15]:
SUMMARY_REGEX = r"^(.*?)\s*\(Summary\)"

NOISE_PATTERNS = [
    r"\s*\(Authors with Filk\)",
    r"\s*\(Recognitions linked to item\)",
    r"\s*\(Author of contact\)",
    r"\s*\(Notes\)",
    r"\s*\(Objectives\)",
    r"\s*\(Most relevant results\)",
    r"\s*\(Degree of originality and innovation\)",
    r"\s*\(Methodological contribution\)",
    r"\s*\(Thematic priority\)",
    r"\s*\(Use of the contribution\)",
    r"\s*\(Cites received for the contribution\)",
    r"\s*\(Quality of the publication/dissemination/exhibition medium\)",
    r"\s*\(Non-academic mentions and references to the contribution\)",
    r"\s*\(Copyright held by the owner/author(s)\)",
]


def clean_expr(col: str):
    expr = pl.col(col)

    for pattern in NOISE_PATTERNS:
        expr = expr.str.replace_all(pattern, "")

    return expr.str.strip_chars()


data_publications = (
    data_researcher.filter(pl.col("dtWorkform") == "Publicaciones")
    .with_columns(
        summary=(
            pl.col("dsDescription").str.extract(SUMMARY_REGEX, 1).str.strip_chars()
        )
    )
    .with_columns(
        description=pl.coalesce(
            [
                pl.col("summary"),
                clean_expr("dsDescription"),
                clean_expr("dsDescription5"),
                pl.col("rfTitle"),
            ]
        ),
    )
    .select("idItem", "dtSubtype", "rfTitle", "description", "rfKeyword")
)

data_publications.to_dicts()

[{'idItem': '10443309',
  'dtSubtype': 'Publicaciones en Revistas',
  'rfTitle': 'ESQAD: A curriculum-aligned dataset for question answer generation in Spanish',
  'description': 'ESQAD: A curriculum-aligned dataset for question answer generation in Spanish. 2026; 335 (): 115255-. DOI:10.1016/j.knosys.2025.115255',
  'rfKeyword': "Adaptive learning; Bloom's taxonomy; Educational technology; Expert systems; Question answer generation"},
 {'idItem': '10429991',
  'dtSubtype': 'Conferencia Publicada',
  'rfTitle': 'Enhancing Public Procurement in the European Union Through Constructing and Exploiting an Integrated Knowledge Graph',
  'description': "Soylu A., SINTEF Foundation for Scientific and Industrial Research; Corcho O., Universidad Politécnica de Madrid; Elvesæter B., SINTEF Foundation for Scientific and Industrial Research; Badenes-Olmedo C., Universidad Politécnica de Madrid; Martínez F.Y., Universidad Politécnica de Madrid; Kovacic M., Jozef Stefan Institute; Posinkovic M., Joze

In [ ]:
import polars as pl
import re

NOISE_MARKERS = [
    r"\s*\(Summary\)",
    r"\s*\(Authors with Filk\)",
    r"\s*\(Recognitions linked to item\)",
    r"\s*\(Author of contact\)",
    r"\s*\(Notes\)",
    r"\s*\(Objectives\)",
    r"\s*\(Most relevant results\)",
    r"\s*\(Degree of originality and innovation\)",
    r"\s*\(Methodological contribution\)",
    r"\s*\(Thematic priority\)",
    r"\s*\(Use of the contribution\)",
    r"\s*\(Cites received for the contribution\)",
    r"\s*\(Quality of the publication/dissemination/exhibition medium\)",
    r"\s*\(Non-academic mentions and references to the contribution\)",
]

CUT_REGEX = r"^(.*?)(?=\s*\((" + "|".join(map(re.escape, NOISE_MARKERS)) + r")\)|$)"


def semantic_clean(col):
    return (
        pl.col(col)
        # extraer solo bloque útil
        .str.extract(CUT_REGEX, 1)
        # espacios raros
        .str.replace_all(r"\s+", " ")
        # limpiar separadores finales
        .str.replace_all(r"[;:,.\s]+$", "")
        # trim
        .str.strip_chars()
    )


data_publications = (
    data_researcher.filter(pl.col("dtWorkform") == "Publicaciones")
    .with_columns(
        desc_main=semantic_clean("dsDescription"),
        desc_alt=semantic_clean("dsDescription5"),
    )
    .with_columns(
        description=pl.coalesce(
            [
                # abstract real
                pl.when(pl.col("desc_main").str.len_chars() > 40).then(
                    pl.col("desc_main")
                ),
                # referencia bibliográfica
                pl.when(pl.col("desc_alt").str.len_chars() > 20).then(
                    pl.col("desc_alt")
                ),
                # fallback
                pl.col("rfTitle"),
            ]
        )
    )
    .select("idItem", "dtSubtype", "rfTitle", "description")
)

ComputeError: regex error: regex parse error:
    ^(.*?)(?=\s*\((\\s\*\\\(Summary\\\)|\\s\*\\\(Authors\ with\ Filk\\\)|\\s\*\\\(Recognitions\ linked\ to\ item\\\)|\\s\*\\\(Author\ of\ contact\\\)|\\s\*\\\(Notes\\\)|\\s\*\\\(Objectives\\\)|\\s\*\\\(Most\ relevant\ results\\\)|\\s\*\\\(Degree\ of\ originality\ and\ innovation\\\)|\\s\*\\\(Methodological\ contribution\\\)|\\s\*\\\(Thematic\ priority\\\)|\\s\*\\\(Use\ of\ the\ contribution\\\)|\\s\*\\\(Cites\ received\ for\ the\ contribution\\\)|\\s\*\\\(Quality\ of\ the\ publication/dissemination/exhibition\ medium\\\)|\\s\*\\\(Non\-academic\ mentions\ and\ references\ to\ the\ contribution\\\))\)|$)
          ^^^
error: look-around, including look-ahead and look-behind, is not supported